# Quiz: Vectors, Matrices, Tensors

Companion self-quiz for `../01_vectors_matrices_tensors.md`.

**How to use**: For each question, predict the answer (shape, error, output) in your head first. Then run the code cell to check. Then read the explanation to confirm your reasoning matches.

Rule: if you got the right answer for the wrong reason, you got it wrong.

In [2]:
import torch

# Standard NLP dimension sizes used throughout this notebook
B, S, D = 2, 4, 8    # batch, sequence length, model dim
H = 2                # num attention heads (so D_h = D/H = 4)
V = 16               # vocab size

/Users/yuhongsun/Library/Python/3.13/lib/python/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## Q1: Tuple syntax — `(D,)` vs `(D)`

What do these two expressions produce? Predict before running.

In [3]:
print(type((D,)), (D,))
print(type((D)),  (D))

<class 'tuple'> (8,)
<class 'int'> 8


**Expected:**
```
<class 'tuple'> (8,)
<class 'int'>   8
```

`(D,)` is a 1-element tuple — the shape of a 1D tensor of length `D`. `(D)` is just `D` in parentheses — a scalar `int`. The trailing comma is load-bearing.

## Q2: What is a `(D,)` tensor?

Create a tensor with shape `(D,)`. How many dimensions does it have?

In [4]:
v = torch.arange(D)
print("shape:", v.shape)
print("ndim:",  v.ndim)

shape: torch.Size([8])
ndim: 1


**Expected:**
```
shape: torch.Size([8])
ndim:  1
```

A 1D tensor of length 8. No 'omitted' second dim — it is genuinely 1D. This is the shape of a bias vector, a LayerNorm scale/shift, a single token embedding, etc.

## Q3: Broadcasting `(D,)` onto `(B, S, D)`

This is the bias-addition pattern. Predict the output shape.

In [5]:
x = torch.zeros(B, S, D)
b = torch.arange(D).float()      # shape (D,) — a bias vector
y = x + b
print("y.shape:", y.shape)
print("y[0, 0]:", y[0, 0])
print("y[1, 3]:", y[1, 3])

y.shape: torch.Size([2, 4, 8])
y[0, 0]: tensor([0., 1., 2., 3., 4., 5., 6., 7.])
y[1, 3]: tensor([0., 1., 2., 3., 4., 5., 6., 7.])


**Expected:**
```
y.shape: torch.Size([2, 4, 8])
y[0, 0]: tensor([0., 1., 2., 3., 4., 5., 6., 7.])
y[1, 3]: tensor([0., 1., 2., 3., 4., 5., 6., 7.])
```

The same `(D,)` bias is added to every `(b, s)` position. Right-align the shapes: `(B, S, D)` vs `(D,)` → missing dims treated as 1 → effectively `(1, 1, D)` broadcasts to `(B, S, D)`.

## Q4: Why `(B, S, D) + (B, D)` fails

What happens if you try to add a per-sequence vector `(B, D)` to `(B, S, D)`?

In [6]:
x = torch.zeros(B, S, D)
per_seq = torch.zeros(B, D)
try:
    y = x + per_seq
    print("succeeded with shape:", y.shape)
except RuntimeError as e:
    print("ERROR:", e)

ERROR: The size of tensor a (4) must match the size of tensor b (2) at non-singleton dimension 1


In [7]:
# Fix: unsqueeze to insert a size-1 dim in the right place
y = x + per_seq.unsqueeze(1)    # (B, D) → (B, 1, D) → broadcasts over S
print("fixed shape:", y.shape)

fixed shape: torch.Size([2, 4, 8])


**Expected:**
- The first cell errors. Right-aligning `(B, S, D)` and `(B, D)` gives `B↔S` and `D↔D`. `B` and `S` don't match and neither is 1, so it fails.
- The second cell produces `torch.Size([2, 4, 8])`. `unsqueeze(1)` turns `(B, D)` into `(B, 1, D)`, which broadcasts correctly.

## Q5: Multi-head attention reshape — split `D` into `(H, D_h)`

Starting from `(B, S, D)`, reshape into per-head form.

In [ ]:
x = torch.arange(B * S * D).reshape(B, S, D).float()
x_heads = x.view(B, S, H, D // H)
print("x:       ", x.shape)
print("x_heads: ", x_heads.shape)

# Sanity check: same underlying memory, no copy
print("shares storage:", x.data_ptr() == x_heads.data_ptr())

**Expected:**
```
x:        torch.Size([2, 4, 8])
x_heads:  torch.Size([2, 4, 2, 4])
shares storage: True
```

`view` reinterprets the same memory — no copy. The `D=8` dim is split into `(H=2, D_h=4)`.

## Q6: Move the head dim next to batch with `permute`

We want `(B, S, H, D_h) → (B, H, S, D_h)` so the attention matmul sees `(S, D_h)` per head.

In [ ]:
x_heads = torch.arange(B * S * H * (D // H)).reshape(B, S, H, D // H).float()
x_perm  = x_heads.permute(0, 2, 1, 3)
print("before: ", x_heads.shape, "contiguous:", x_heads.is_contiguous())
print("after:  ", x_perm.shape,  "contiguous:", x_perm.is_contiguous())

**Expected:**
```
before:  torch.Size([2, 4, 2, 4]) contiguous: True
after:   torch.Size([2, 2, 4, 4]) contiguous: False
```

`permute` only rewrites strides; the underlying memory is unchanged, so the result is non-contiguous. This is the setup for the classic `view`-after-`permute` gotcha.

## Q7: The `view`-after-`permute` footgun

Try to merge heads back with `view`. What happens?

In [ ]:
x = torch.arange(B * S * D).reshape(B, S, D).float()
x_h = x.view(B, S, H, D // H).permute(0, 2, 1, 3)    # (B, H, S, D_h), non-contiguous

try:
    merged = x_h.view(B, S, D)
except RuntimeError as e:
    print("ERROR:", str(e)[:200])

In [ ]:
# Fix option A: reshape handles non-contiguous (falls back to copy if needed)
merged_a = x_h.reshape(B, S, D)
print("reshape worked:", merged_a.shape)

# Fix option B: .contiguous() forces a copy, then view is fine
# NOTE: to merge (B, H, S, D_h) back into (B, S, D), you need to permute
# head and sequence axes back first — otherwise you'd merge the wrong dims.
merged_b = x_h.contiguous().view(B, H, S, D // H).permute(0, 2, 1, 3).reshape(B, S, D)
print("contig.view worked:", merged_b.shape)

**Expected:**
- First cell errors with something like: `view size is not compatible with input tensor's size and stride`. `view` requires contiguous memory when merging dims.
- Second cell works. `reshape` silently falls back to a copy when needed.

**Rule of thumb**: use `reshape` by default. Use `view` only when you want it to scream if the layout is wrong.

## Q8: Attention mask broadcasting

You have attention scores `(B, H, S, S)` and a padding mask `(B, S)` marking which keys are padding (True = mask out). How do you add the mask to scores so it broadcasts correctly across heads and across query positions?

In [ ]:
scores = torch.zeros(B, H, S, S)
pad_mask = torch.tensor([[False, False, True, True],
                         [False, True,  True, True]])    # (B, S)
print("pad_mask.shape:", pad_mask.shape)

# Wrong way: directly add (B, S) to (B, H, S, S) — right-aligns incorrectly
try:
    bad = scores + pad_mask.float()
    print("naive add succeeded, shape:", bad.shape, "(but mask is in the wrong place!)")
except RuntimeError as e:
    print("naive add ERROR:", e)

In [ ]:
# Right way: reshape (B, S) → (B, 1, 1, S) so it broadcasts over H and over queries
mask_bias = pad_mask.float().masked_fill(pad_mask, float('-inf')).masked_fill(~pad_mask, 0.0)
mask_bias = mask_bias.view(B, 1, 1, S)
print("mask_bias.shape:", mask_bias.shape)

masked_scores = scores + mask_bias
print("masked_scores.shape:", masked_scores.shape)
print("scores[0, 0] (batch 0, head 0, all queries, all keys):")
print(masked_scores[0, 0])

**Expected:**
- The naive add might or might not error depending on shapes, but it's semantically wrong — broadcasting places the mask on the wrong axis.
- The correct mask shape is `(B, 1, 1, S)`:
  - `B` matches the batch.
  - First `1` broadcasts across all heads (same padding for every head).
  - Second `1` broadcasts across all query positions (every query sees the same padded keys).
  - `S` is the key dimension — the one we actually mask.
- `masked_scores[0, 0]` should have `-inf` in columns 2 and 3 for every row.

## Q9: Why `q @ k.transpose(-1, -2)` produces `(B, H, S, S)`

Walk through the shapes.

In [ ]:
D_h = D // H
q = torch.randn(B, H, S, D_h)
k = torch.randn(B, H, S, D_h)

print("q:           ", q.shape)
print("k:           ", k.shape)
print("k.transpose: ", k.transpose(-1, -2).shape)

scores = q @ k.transpose(-1, -2)
print("scores:      ", scores.shape)

**Expected:**
```
q:            torch.Size([2, 2, 4, 4])
k:            torch.Size([2, 2, 4, 4])
k.transpose:  torch.Size([2, 2, 4, 4])
scores:       torch.Size([2, 2, 4, 4])
```

Batched matmul: the leading `(B, H)` dims are broadcast/batched. The last two dims obey matmul rules: `(S, D_h) @ (D_h, S) → (S, S)`. So the result is `(B, H, S, S)` — one `S×S` attention matrix per (batch, head).

## Q10: `expand` vs `repeat` — memory semantics

Both can tile a size-1 dim. What's the difference in memory behavior?

In [ ]:
v = torch.arange(D).float()             # shape (D,)
v_unsq = v.unsqueeze(0).unsqueeze(0)     # (1, 1, D)

exp = v_unsq.expand(B, S, D)
rep = v_unsq.repeat(B, S, 1)

print("expand shape:", exp.shape, "shares storage:", exp.data_ptr() == v.data_ptr())
print("repeat shape:", rep.shape, "shares storage:", rep.data_ptr() == v.data_ptr())

print("\nexpand strides (zero means broadcasting):", exp.stride())
print("repeat strides:                          ", rep.stride())

**Expected:**
```
expand shape: torch.Size([2, 4, 8]) shares storage: True
repeat shape: torch.Size([2, 4, 8]) shares storage: False

expand strides (zero means broadcasting): (0, 0, 1)
repeat strides:                           (32, 8, 1)
```

`expand` is free — it sets strides to 0 along broadcast dims, reading the same memory multiple times. `repeat` allocates `B*S*D` new floats and copies. Prefer `expand` unless you need independent writable memory.

## Q11: Flattening for cross-entropy

`F.cross_entropy` wants `(N, V)` logits and `(N,)` labels. Reshape from `(B, S, V)` logits and `(B, S)` labels.

In [ ]:
import torch.nn.functional as F

logits = torch.randn(B, S, V)
labels = torch.randint(0, V, (B, S))

# Reshape for cross_entropy
flat_logits = logits.reshape(-1, V)     # (B*S, V)
flat_labels = labels.reshape(-1)        # (B*S,)

print("flat_logits:", flat_logits.shape)
print("flat_labels:", flat_labels.shape)

loss = F.cross_entropy(flat_logits, flat_labels)
print("loss:", loss.item())

**Expected:**
```
flat_logits: torch.Size([8, 16])
flat_labels: torch.Size([8,])
loss: (some scalar around log(V) = log(16) ≈ 2.77 for random logits)
```

`B*S = 8` independent classification problems, each over `V = 16` classes. Equivalent idiom: `logits.flatten(0, 1)`.

## Q12: Silent broadcast bug

This bug is the reason you should always check shapes. What's the output shape, and why is it a bug?

In [ ]:
# Suppose attention scores are (B, H, S, S)
# You want to add a per-query bias of shape (S,) — one scalar per query position
scores = torch.zeros(B, H, S, S)
query_bias = torch.arange(S).float()     # shape (S,)

result = scores + query_bias
print("result.shape:", result.shape)
print("result[0, 0]:")
print(result[0, 0])

**Expected:**
```
result.shape: torch.Size([2, 2, 4, 4])
result[0, 0]:
tensor([[0., 1., 2., 3.],
        [0., 1., 2., 3.],
        [0., 1., 2., 3.],
        [0., 1., 2., 3.]])
```

**The bug**: you intended a per-query bias (vary across rows, same across columns), but right-aligning `(S,)` to `(B, H, S, S)` places it on the **last** dim — which is the **key** dim. So the bias varies across columns, not rows. Every query got the same bias, every key got a different one. The opposite of what you wanted.

**Fix**: be explicit — `query_bias.view(1, 1, S, 1)` to put it on the query axis.

In [ ]:
result_correct = scores + query_bias.view(1, 1, S, 1)
print("result_correct[0, 0]:")
print(result_correct[0, 0])

**Expected (correct):**
```
tensor([[0., 0., 0., 0.],
        [1., 1., 1., 1.],
        [2., 2., 2., 2.],
        [3., 3., 3., 3.]])
```

Now the bias varies across rows (queries) and is constant across columns (keys) — per-query bias. **Moral**: when broadcasting a rank-1 tensor into a rank-4 one, never rely on right-alignment to 'do the right thing' — reshape explicitly to the target rank with `1`s in the dims you want to broadcast over. It documents intent and prevents this class of bug.

## Scoring

- **12/12**: solid — move on to the next file.
- **9–11**: skim the file's sections for the ones you missed.
- **≤ 8**: reread the file before moving on. The shape gymnastics in this file show up in every subsequent topic (attention, MHA, multi-query, flash-attention, RoPE).